# Task 1: Build a Complete ETL Pipeline


In [1]:
#step 1
import pandas as pd 
import random
from datetime import datetime,timedelta


random.seed(19)
product_names = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones","Phone", "Tablet", "Camera", "Printer", "Speaker"]
shipping_countries=[ "USA", "Germany", "Canada", "France", "Japan","Brazil", "Italy", "Spain", "Australia", "Turkey"]
raw_orders=[]

for i in range(1,201):
    order_records={
        "order_id": i,
        "customer_id": random.randint(1000, 1100),
        "product_name": random.choice(product_names),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 500), 2),
        "order_date": str(
            pd.Timestamp("2025-01-01") + pd.Timedelta(days=random.randint(0, 60))
        )[:10],
        "shipping_country": random.choice(shipping_countries)
    }
    raw_orders.append(order_records)

# 3 replaced with space 
raw_orders[0]["product_name"] = None
raw_orders[1]["product_name"] = ""
raw_orders[2]["product_name"] = None

# 3 negative quantity or unit_price
raw_orders[3]["quantity"] = -2
raw_orders[4]["unit_price"] = -50
raw_orders[5]["quantity"] = -1

# 3 malformed order_date
raw_orders[6]["order_date"] = "not-a-date"
raw_orders[7]["order_date"] = "2025-13-45"
raw_orders[8]["order_date"] = "31-02-2025"

# 3 duplicate order_id
raw_orders[9]["order_id"] = raw_orders[0]["order_id"]
raw_orders[10]["order_id"] = raw_orders[1]["order_id"]
raw_orders[11]["order_id"] = raw_orders[2]["order_id"]

# 3 quantity or unit_price as string
raw_orders[12]["quantity"] = "3"
raw_orders[13]["unit_price"] = "45.99"
raw_orders[14]["quantity"] = "two"

In [2]:
def extract(raw_records):
    valid_records = []
    rejected_records = []
    reason_count = {}
    seen_ids = set()

    for r in raw_records:
        reason = None

        # duplicate check
        if r["order_id"] in seen_ids:
            reason = "duplicate_id"
        seen_ids.add(r["order_id"])

        # missing product_name
        if reason is None:
            if r["product_name"] is None or r["product_name"] == "":
                reason = "missing_field"

        # number check
        if reason is None:
            if not isinstance(r["quantity"], (int, float)) or not isinstance(r["unit_price"], (int, float)):
                reason = "not_numeric"
            elif r["quantity"] <= 0 or r["unit_price"] <= 0:
                reason = "non_positive_number"

        # date check
        if reason is None:
            try:
                datetime.strptime(r["order_date"], "%Y-%m-%d")
            except:
                reason = "invalid_date"

        if reason is None:
            valid_records.append(r)
        else:
            r2 = r.copy()
            r2["reason"] = reason
            rejected_records.append(r2)
            reason_count[reason] = reason_count.get(reason, 0) + 1

    print("Valid records:", len(valid_records))
    print("Rejected records:", len(rejected_records))
    print("Rejected by reason:", reason_count)

    return valid_records, rejected_records


valid_records, rejected_records = extract(raw_orders)

Valid records: 185
Rejected records: 15
Rejected by reason: {'missing_field': 3, 'non_positive_number': 3, 'invalid_date': 3, 'duplicate_id': 3, 'not_numeric': 3}


In [3]:
#step 1
def transform(valid_records):
    df = pd.DataFrame(valid_records)
    # total amount
    df["total_amount"] = df["quantity"] * df["unit_price"]
    
    # converting date
    df["order_date"] = pd.to_datetime(df["order_date"])

    # extracting month and day of week
    df["order_month"] = df["order_date"].dt.month
    df["order_day_of_week"] = df["order_date"].dt.day_name()

    # standardizing country
    df["shipping_country"] = df["shipping_country"].str.title()

    # removing duplicate order_id
    df = df.drop_duplicates(subset="order_id", keep="first")

    # amount category
    def category(x):
        if x < 25:
            return "small"
        elif x <= 100:
            return "medium"
        else:
            return "large"

    df["amount_category"] = df["total_amount"].apply(category)
    return df


clean_df = transform(valid_records)
clean_df.head()

,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country,total_amount,order_month,order_day_of_week,amount_category
0,16,1009,Monitor,3,346.86,2025-02-18,Italy,1040.58,2,Tuesday,large
1,17,1012,Camera,1,239.67,2025-01-29,Usa,239.67,1,Wednesday,large
2,18,1022,Tablet,3,46.93,2025-01-09,Canada,140.79,1,Thursday,large
3,19,1055,Headphones,4,417.51,2025-01-13,Turkey,1670.04,1,Monday,large
4,20,1060,Mouse,4,327.76,2025-02-10,Canada,1311.04,2,Monday,large


In [4]:
#step 3
def load(df, path):

    df.to_parquet(path, index=False)

    df2 = pd.read_parquet(path)

    print("Original rows:", len(df))
    print("Loaded rows:", len(df2))

    print("\nOriginal dtypes:")
    print(df.dtypes)

    print("\nLoaded dtypes:")
    print(df2.dtypes)

    return df2


loaded_df = load(clean_df, "orders.parquet")

Original rows: 185
Loaded rows: 185

Original dtypes:
order_id                      int64
customer_id                   int64
product_name                    str
quantity                      int64
unit_price                  float64
order_date           datetime64[us]
shipping_country                str
total_amount                float64
order_month                   int32
order_day_of_week               str
amount_category                 str
dtype: object

Loaded dtypes:
order_id                      int64
customer_id                   int64
product_name                    str
quantity                      int64
unit_price                  float64
order_date           datetime64[us]
shipping_country                str
total_amount                float64
order_month                   int32
order_day_of_week               str
amount_category                 str
dtype: object


# Task 2: ETL vs ELT Comparison


In [5]:
import pandas as pd
import json

#Extracting
def extract_elt(raw_records):
    accepted = []
    rejected = []

    for r in raw_records:
        if isinstance(r, dict):
            accepted.append(r)
        else:
            rejected.append({"record": r, "reason": "unparseable_record"})

    print("Accepted:", len(accepted))
    print("Rejected:", len(rejected))
    return accepted, rejected

#Loading
def load_elt(raw_data, path):
    df = pd.DataFrame({
        "raw_record": [json.dumps(r) for r in raw_data]
    })
    df.to_parquet(path, index=False)
    print(f"Loaded {len(df)} raw records to {path}")


#Transforming
def transform_elt(path):
    df = pd.read_parquet(path)
    records = [json.loads(x) for x in df["raw_record"]]

    valid_records = []
    seen_ids = set()

    for r in records:
        reason = None

        if r["order_id"] in seen_ids:
            reason = "duplicate_id"
        seen_ids.add(r["order_id"])

        if reason is None:
            if r["product_name"] is None or r["product_name"] == "":
                reason = "missing_field"

        if reason is None:
            if not isinstance(r["quantity"], (int, float)) or not isinstance(r["unit_price"], (int, float)):
                reason = "not_numeric"
            elif r["quantity"] <= 0 or r["unit_price"] <= 0:
                reason = "non_positive_number"

        if reason is None:
            try:
                pd.to_datetime(r["order_date"])
            except:
                reason = "invalid_date"

        if reason is None:
            valid_records.append(r)

    df = pd.DataFrame(valid_records)

    df["total_amount"] = df["quantity"] * df["unit_price"]
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["order_month"] = df["order_date"].dt.month
    df["order_day_of_week"] = df["order_date"].dt.day_name()
    df["shipping_country"] = df["shipping_country"].str.title()

    df["amount_category"] = "large"
    df.loc[df["total_amount"] < 25, "amount_category"] = "small"
    df.loc[(df["total_amount"] >= 25) & (df["total_amount"] <= 100), "amount_category"] = "medium"

    return df


accepted_records, rejected_records = extract_elt(raw_orders)
load_elt(accepted_records, "raw_orders_lake.parquet")

clean_df = transform_elt("raw_orders_lake.parquet")
print("Final clean row count:", len(clean_df))
clean_df.head()

Accepted: 200
Rejected: 0
Loaded 200 raw records to raw_orders_lake.parquet
Final clean row count: 185


,order_id,customer_id,product_name,quantity,unit_price,order_date,shipping_country,total_amount,order_month,order_day_of_week,amount_category
0,16,1009,Monitor,3,346.86,2025-02-18,Italy,1040.58,2,Tuesday,large
1,17,1012,Camera,1,239.67,2025-01-29,Usa,239.67,1,Wednesday,large
2,18,1022,Tablet,3,46.93,2025-01-09,Canada,140.79,1,Thursday,large
3,19,1055,Headphones,4,417.51,2025-01-13,Turkey,1670.04,1,Monday,large
4,20,1060,Mouse,4,327.76,2025-02-10,Canada,1311.04,2,Monday,large


#### How many records made it to the destination in each approach?
At the end of both processes, we obtained 185 cleaned records, because both pipelines used the same dataset and the same validation rules.

#### At what stage were problems caught in each?
In the **ETL** approach, the problems were caught in the **Extract stage**.During this stage we checked the data and validated it before loading. If a record had problems,for example missing product name, wrong date, negative values, or duplicate ID, it was rejected immediately. So the bad records were removed before the data was loaded.

In the **ELT** approach, the problems were caught in the **Transform stage**. In this pipeline we first loaded the raw data into the data lake with almost no validation. After loading the data, we applied the validation and cleaning rules during the transformation step. At that stage the incorrect records were detected and removed.

#### What are the advantages of each approach?

The **ETL** approach has the advantage that the data is cleaned before it is loaded and this means the system only stores valid and trusted data. It also reduces the risk of incorrect data being stored in the database and can make the final dataset easier to manage.In the **ELT** approach, the raw data is stored first and the cleaning happens later. This allows the original data to be preserved, which can be useful if we want to analyze it in different ways or apply new transformations in the future. It is also more flexible and works well when dealing with large amounts of data in modern data lake environments.
#### When would you choose one over the other?
ETL is typically chosen when data needs to be cleaned and validated before it is stored, especially in situations where data quality and consistency are very important. ELT is usually preferred when working with large datasets or modern data platforms, because raw data can be stored first and then transformed later depending on the analysis requirements.

# Task 3: Simulate Modes of Dataflow


### step 1

In [6]:
# Step 1
# Shared database (simulated)
database = {"orders": [], "features": {}}

def order_service_write_db(order):
    database["orders"].append(order)

def feature_service_compute_db():
    features = {}

    for order in database["orders"]:
        cid = order["customer_id"]
        amount = order["quantity"] * order["unit_price"]
        date = order["order_date"]

        if cid not in features:
            features[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "last_order_date": date
            }

        features[cid]["total_orders"] += 1
        features[cid]["total_amount"] += amount

        if date > features[cid]["last_order_date"]:
            features[cid]["last_order_date"] = date

    for cid in features:
        features[cid]["avg_amount"] = (
            features[cid]["total_amount"] /
            features[cid]["total_orders"]
        )
        del features[cid]["total_amount"]

    database["features"] = features

def prediction_service_read_db(customer_id):
    return database["features"].get(customer_id)


# example orders
order_service_write_db({
    "customer_id": 1,
    "quantity": 2,
    "unit_price": 50,
    "order_date": "2025-03-01"
})

order_service_write_db({
    "customer_id": 1,
    "quantity": 1,
    "unit_price": 30,
    "order_date": "2025-03-05"
})

order_service_write_db({
    "customer_id": 2,
    "quantity": 3,
    "unit_price": 20,
    "order_date": "2025-03-02"
})

feature_service_compute_db()

print(prediction_service_read_db(1))

{'total_orders': 2, 'last_order_date': '2025-03-05', 'avg_amount': 65.0}


### Step 2

In [7]:
# Order service
orders = []

def order_service_write_api(order):
    orders.append(order)

def order_service_get_orders_api():
    return orders


# Feature service
def feature_service_compute_api():
    orders_local = order_service_get_orders_api()
    features = {}

    for order in orders_local:
        cid = order["customer_id"]
        amount = order["quantity"] * order["unit_price"]
        date = order["order_date"]

        if cid not in features:
            features[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "last_order_date": date
            }

        features[cid]["total_orders"] += 1
        features[cid]["total_amount"] += amount

        if date > features[cid]["last_order_date"]:
            features[cid]["last_order_date"] = date

    for cid in features:
        features[cid]["avg_amount"] = (
            features[cid]["total_amount"] /
            features[cid]["total_orders"]
        )
        del features[cid]["total_amount"]

    return features


# Prediction service
def prediction_service_read_api(customer_id):
    features = feature_service_compute_api()
    return features.get(customer_id)


order_service_write_api({"customer_id": 101, "quantity": 2, "unit_price": 25, "order_date": "2025-03-01"})
order_service_write_api({"customer_id": 101, "quantity": 1, "unit_price": 50, "order_date": "2025-03-08"})

print(prediction_service_read_api(101))

{'total_orders': 2, 'last_order_date': '2025-03-08', 'avg_amount': 50.0}


### step 3

In [8]:
# Step 3 

class Broker:
    def __init__(self):
        self.topics = {}

    def publish(self, topic, message):
        if topic not in self.topics:
            self.topics[topic] = []
        self.topics[topic].append(message)

    def subscribe(self, topic):
        return self.topics.get(topic, [])

    def get_latest(self, topic):
        if topic in self.topics and self.topics[topic]:
            return self.topics[topic][-1]
        return None


broker = Broker()


# Order service
def order_service_write_broker(order):
    broker.publish("orders", order)


# Feature service
def feature_service_compute_broker():
    orders = broker.subscribe("orders")
    features = {}

    for order in orders:
        cid = order["customer_id"]
        amount = order["quantity"] * order["unit_price"]
        date = order["order_date"]

        if cid not in features:
            features[cid] = {
                "total_orders": 0,
                "total_amount": 0,
                "last_order_date": date
            }

        features[cid]["total_orders"] += 1
        features[cid]["total_amount"] += amount

        if date > features[cid]["last_order_date"]:
            features[cid]["last_order_date"] = date

    for cid in features:
        features[cid]["avg_amount"] = (
            features[cid]["total_amount"] /
            features[cid]["total_orders"]
        )
        del features[cid]["total_amount"]

    broker.publish("features", features)


# Prediction service
def prediction_service_read_broker(customer_id):
    features = broker.subscribe("features")
    if features:
        latest_features = features[-1]
        return latest_features.get(customer_id)
    return None


# Example
order_service_write_broker({
    "customer_id": 101,
    "quantity": 1,
    "unit_price": 50,
    "order_date": "2025-03-08"
})

order_service_write_broker({
    "customer_id": 102,
    "quantity": 3,
    "unit_price": 20,
    "order_date": "2025-03-04"
})

feature_service_compute_broker()

print(prediction_service_read_broker(101))
print(prediction_service_read_broker(102))

{'total_orders': 1, 'last_order_date': '2025-03-08', 'avg_amount': 50.0}
{'total_orders': 1, 'last_order_date': '2025-03-04', 'avg_amount': 60.0}


### step 4

In [9]:
#step 4 for step 1

random.seed(42)

# create the same 20 orders for all three modes
orders_20 = []
for i in range(20):
    orders_20.append({
        "customer_id": random.randint(1, 5),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 100), 2),
        "order_date": (datetime.now() - timedelta(days=random.randint(0, 10))).strftime("%Y-%m-%d")
    })


database = {"orders": [], "features": {}}

for order in orders_20:
    order_service_write_db(order)

feature_service_compute_db()

step1_results = {}
for cid in range(1, 6):
    step1_results[cid] = prediction_service_read_db(cid)

print("STEP 1 RESULTS")
for cid, features in step1_results.items():
    print(cid, ":", features)


STEP 1 RESULTS
1 : {'total_orders': 5, 'last_order_date': '2026-03-07', 'avg_amount': 123.08999999999999}
2 : {'total_orders': 2, 'last_order_date': '2026-03-01', 'avg_amount': 98.385}
3 : {'total_orders': 6, 'last_order_date': '2026-03-08', 'avg_amount': 98.32833333333333}
4 : {'total_orders': 2, 'last_order_date': '2026-03-05', 'avg_amount': 105.59}
5 : {'total_orders': 5, 'last_order_date': '2026-03-08', 'avg_amount': 95.662}


In [10]:
#step 4 for step 2

orders = []

for order in orders_20:
    order_service_write_api(order)

step2_results = {}
for cid in range(1, 6):
    step2_results[cid] = prediction_service_read_api(cid)

print("\nSTEP 2 RESULTS")
for cid, features in step2_results.items():
    print(cid, ":", features)


STEP 2 RESULTS
1 : {'total_orders': 5, 'last_order_date': '2026-03-07', 'avg_amount': 123.08999999999999}
2 : {'total_orders': 2, 'last_order_date': '2026-03-01', 'avg_amount': 98.385}
3 : {'total_orders': 6, 'last_order_date': '2026-03-08', 'avg_amount': 98.32833333333333}
4 : {'total_orders': 2, 'last_order_date': '2026-03-05', 'avg_amount': 105.59}
5 : {'total_orders': 5, 'last_order_date': '2026-03-08', 'avg_amount': 95.662}


In [11]:
#step 4 for step 3
broker = Broker()

for order in orders_20:
    order_service_write_broker(order)

feature_service_compute_broker()

step3_results = {}
for cid in range(1, 6):
    step3_results[cid] = prediction_service_read_broker(cid)

print("\nSTEP 3 RESULTS")
for cid, features in step3_results.items():
    print(cid, ":", features)



STEP 3 RESULTS
1 : {'total_orders': 5, 'last_order_date': '2026-03-07', 'avg_amount': 123.08999999999999}
2 : {'total_orders': 2, 'last_order_date': '2026-03-01', 'avg_amount': 98.385}
3 : {'total_orders': 6, 'last_order_date': '2026-03-08', 'avg_amount': 98.32833333333333}
4 : {'total_orders': 2, 'last_order_date': '2026-03-05', 'avg_amount': 105.59}
5 : {'total_orders': 5, 'last_order_date': '2026-03-08', 'avg_amount': 95.662}


In [12]:
#verification part
print("\nStep 1 == Step 2:", step1_results == step2_results)
print("Step 2 == Step 3:", step2_results == step3_results)
print("All three match:", step1_results == step2_results == step3_results)


Step 1 == Step 2: True
Step 2 == Step 3: True
All three match: True


### comparing
After running the same 20 orders through the three different dataflow designs, the prediction outputs remain the same because the logic used to compute the features does not change; only the way the data moves between services is different. In the database-based design the services communicate through shared storage, so they are less directly connected, but this also means every step depends on reading and writing to the same place, which can make the process slower. 

A direct service-to-service design usually has lower latency because the prediction service can immediately request results from the feature service, but the services become tightly connected, so if the feature service stops working the prediction service cannot continue. The message broker design works differently because each service only sends or receives messages from topics, which reduces direct dependency between components.

In this setup, if one service temporarily stops, the messages can still remain in the broker and be processed later. Because of this, the three modes produce the same analytical result but behave very differently in terms of coupling, response speed, and system reliability when something fails.

# Task 4: Batch Processing vs Stream Processing

In [13]:
# Step 1 — Batch processing

def daily_batch_features(df):

    import pandas as pd

    # computing revenue
    df["revenue"] = df["quantity"] * df["unit_price"]

    # daily aggregates
    daily_stats = df.groupby("order_date").agg(
        total_orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_size=("revenue", "mean"),
        unique_customers=("customer_id", "nunique")
    ).reset_index()

    # top product by revenue per day
    product_rev = df.groupby(["order_date","product_name"])["revenue"].sum().reset_index()

    top_products = (
        product_rev.sort_values(["order_date","revenue"], ascending=[True,False])
        .drop_duplicates("order_date")
        [["order_date","product_name"]]
        .rename(columns={"product_name":"top_product"})
    )

    # merge results
    result = daily_stats.merge(top_products, on="order_date")

    return result

# run batch processing
batch_results = daily_batch_features(clean_df)
print(batch_results)

batch_total_orders = batch_results["total_orders"].sum()
batch_total_revenue = batch_results["total_revenue"].sum()

print("\nBatch orders:", batch_total_orders)
print("Batch revenue:", batch_total_revenue)

   order_date  total_orders  total_revenue  avg_order_size  unique_customers  \
0  2025-01-01             4        3560.71      890.177500                 4   
1  2025-01-02             2         910.75      455.375000                 2   
2  2025-01-03             1          76.92       76.920000                 1   
3  2025-01-04             2        1854.78      927.390000                 2   
4  2025-01-05             2        1953.42      976.710000                 2   
5  2025-01-06             5        5329.93     1065.986000                 5   
6  2025-01-07             2        2161.30     1080.650000                 2   
7  2025-01-08             3        1258.24      419.413333                 3   
8  2025-01-09             5        4628.47      925.694000                 4   
9  2025-01-10             3        2240.61      746.870000                 3   
10 2025-01-11             2         721.20      360.600000                 2   
11 2025-01-12             2         774.

In [14]:
# step 2

from collections import deque, Counter

class StreamProcessor:
    def __init__(self, window_size=50):
        self.window_size = window_size

        self.total_orders = 0
        self.total_revenue = 0
        self.unique_customers = set()

        self.last_orders = deque()
        self.product_counts = Counter()

    def process_order(self, order):
        amount = order["quantity"] * order["unit_price"]
        product = order["product_name"]
        customer = order["customer_id"]

        # running totals
        self.total_orders += 1
        self.total_revenue += amount
        self.unique_customers.add(customer)

        # add to window
        self.last_orders.append({"amount": amount, "product_name": product})
        self.product_counts[product] += 1

        # keep only last 50 orders
        if len(self.last_orders) > self.window_size:
            old_order = self.last_orders.popleft()
            old_product = old_order["product_name"]
            self.product_counts[old_product] -= 1
            if self.product_counts[old_product] == 0:
                del self.product_counts[old_product]

    def get_stats(self):
        if len(self.last_orders) > 0:
            window_avg = sum(order["amount"] for order in self.last_orders) / len(self.last_orders)
            popular_product = self.product_counts.most_common(1)[0][0]
        else:
            window_avg = 0
            popular_product = None

        return {
            "running_total_orders": self.total_orders,
            "running_total_revenue": self.total_revenue,
            "window_avg_order_size": window_avg,
            "running_unique_customers": len(self.unique_customers),
            "current_most_popular_product": popular_product
        }

# process the same cleaned data one record at a time
stream = StreamProcessor(window_size=50)

for i, order in clean_df.iterrows():
    stream.process_order(order)

    if (i + 1) % 50 == 0:
        print(f"After {i+1} records:")
        print(stream.get_stats())
        print()

print("Stream orders:", stream.total_orders)
print("Stream revenue:", stream.total_revenue)

After 50 records:
{'running_total_orders': 50, 'running_total_revenue': 29636.4, 'window_avg_order_size': 592.7280000000001, 'running_unique_customers': 41, 'current_most_popular_product': 'Printer'}

After 100 records:
{'running_total_orders': 100, 'running_total_revenue': 79738.17000000003, 'window_avg_order_size': 1002.0354, 'running_unique_customers': 64, 'current_most_popular_product': 'Camera'}

After 150 records:
{'running_total_orders': 150, 'running_total_revenue': 114851.48000000004, 'window_avg_order_size': 702.2661999999999, 'running_unique_customers': 78, 'current_most_popular_product': 'Monitor'}

Stream orders: 185
Stream revenue: 148868.65000000008


### step 3
After processing the dataset using both approaches, the final cumulative statistics match. The total number of orders and the total revenue produced by the batch processing pipeline are the same as those produced by the streaming processor. This shows that both methods process the same data correctly, even though they operate differently.

However, windowed streaming statistics can differ from daily batch statistics because streaming calculations are based on a moving window (for example, the last 50 orders) rather than a full daily dataset. As new orders arrive, older ones leave the window, so the metrics can change continuously.

Batch processing provides stable summaries over a fixed time period and is useful for historical reporting and analysis. In contrast, streaming processing provides real-time insights and allows systems to detect short-term trends or changes as events occur. Together, they provide both reliable historical aggregates and immediate operational visibility.

# Task 5: Combine Batch and Stream Features

In [15]:
# step 1

import pandas as pd
from datetime import datetime

# Batch features function
def compute_batch_features(df, customer_id):

    customer_orders = df[df["customer_id"] == customer_id]

    total_lifetime_orders = len(customer_orders)
    avg_order_amount = customer_orders["total_amount"].mean()
    days_since_first_order = (datetime.now() - customer_orders["order_date"].min()).days
    most_purchased_category = customer_orders["product_name"].mode()[0]

    return {
        "total_lifetime_orders": total_lifetime_orders,
        "avg_order_amount": avg_order_amount,
        "days_since_first_order": days_since_first_order,
        "most_purchased_category": most_purchased_category
    }


# Streaming features function (last 10 orders)
def compute_streaming_features(df, customer_id, window=10):

    customer_orders = df[df["customer_id"] == customer_id].sort_values("order_date")

    last_orders = customer_orders.tail(window)

    recent_order_count = len(last_orders)
    recent_avg_amount = last_orders["total_amount"].mean()
    seconds_since_last_order = (
        datetime.now() - last_orders["order_date"].max()
    ).total_seconds()
    recent_top_category = last_orders["product_name"].mode()[0]

    return {
        "recent_order_count": recent_order_count,
        "recent_avg_amount": recent_avg_amount,
        "seconds_since_last_order": seconds_since_last_order,
        "recent_top_category": recent_top_category
    }

In [16]:
# Step 2

clean_df["order_date"] = pd.to_datetime(clean_df["order_date"])

# we choose 5 sample customers
sample_customers = clean_df["customer_id"].drop_duplicates().head(5)

batch_features_all = {}
streaming_features_all = {}

for customer_id in sample_customers:
    batch_features_all[customer_id] = compute_batch_features(clean_df, customer_id)
    streaming_features_all[customer_id] = compute_streaming_features(clean_df, customer_id)

print("Batch features:")
for customer_id, features in batch_features_all.items():
    print(f"Customer {customer_id}: {features}")

print("\nStreaming features:")
for customer_id, features in streaming_features_all.items():
    print(f"Customer {customer_id}: {features}")

Batch features:
Customer 1009: {'total_lifetime_orders': 1, 'avg_order_amount': np.float64(1040.58), 'days_since_first_order': 383, 'most_purchased_category': 'Monitor'}
Customer 1012: {'total_lifetime_orders': 1, 'avg_order_amount': np.float64(239.67), 'days_since_first_order': 403, 'most_purchased_category': 'Camera'}
Customer 1022: {'total_lifetime_orders': 2, 'avg_order_amount': np.float64(141.26999999999998), 'days_since_first_order': 423, 'most_purchased_category': 'Tablet'}
Customer 1055: {'total_lifetime_orders': 3, 'avg_order_amount': np.float64(1113.36), 'days_since_first_order': 419, 'most_purchased_category': 'Headphones'}
Customer 1060: {'total_lifetime_orders': 3, 'avg_order_amount': np.float64(1071.2133333333334), 'days_since_first_order': 413, 'most_purchased_category': 'Camera'}

Streaming features:
Customer 1009: {'recent_order_count': 1, 'recent_avg_amount': np.float64(1040.58), 'seconds_since_last_order': 33163936.132243, 'recent_top_category': 'Monitor'}
Customer 1

In [17]:
# step 3

combined_features = {}

for customer_id in sample_customers:
    combined_features[customer_id] = {
        **batch_features_all[customer_id],
        **streaming_features_all[customer_id]
    }

for cid, features in combined_features.items():
    print(f"\nCustomer {cid}")
    for k, v in features.items():
        print(f"  {k}: {v}")


Customer 1009
  total_lifetime_orders: 1
  avg_order_amount: 1040.58
  days_since_first_order: 383
  most_purchased_category: Monitor
  recent_order_count: 1
  recent_avg_amount: 1040.58
  seconds_since_last_order: 33163936.132243
  recent_top_category: Monitor

Customer 1012
  total_lifetime_orders: 1
  avg_order_amount: 239.67
  days_since_first_order: 403
  most_purchased_category: Camera
  recent_order_count: 1
  recent_avg_amount: 239.67
  seconds_since_last_order: 34891936.137379
  recent_top_category: Camera

Customer 1022
  total_lifetime_orders: 2
  avg_order_amount: 141.26999999999998
  days_since_first_order: 423
  most_purchased_category: Tablet
  recent_order_count: 2
  recent_avg_amount: 141.26999999999998
  seconds_since_last_order: 32991136.142623
  recent_top_category: Tablet

Customer 1055
  total_lifetime_orders: 3
  avg_order_amount: 1113.36
  days_since_first_order: 419
  most_purchased_category: Headphones
  recent_order_count: 3
  recent_avg_amount: 1113.36
  se

### step 4

A machine learning model works better when it uses both batch features and stream features because they show different types of information about a customer. Batch features describe long-term behavior, such as how many orders a customer has made in total or what category they usually buy. This helps the model understand stable habits. Stream features describe very recent activity, like what the customer did in the last few orders, so the model can react to changes quickly. For example, if the goal is to predict whether a user will make another purchase in the next few minutes, batch features alone are not enough because they cannot show the user’s current actions during the session.

On the other hand, if the goal is to predict a customer’s long-term value for the company, stream features alone are not enough because they only look at a small number of recent orders and ignore the full purchase history. Using both types of features helps the model understand both the past behavior and the current situation.